# 11 · Advanced Backtracking: N-Queens & Rat in a Maze

**Backtracking** = build a solution one step at a time, and as soon as a partial solution cannot work, **undo the last step** and try the next option.

The template used in both problems:
```
solve(state):
    if state is a complete solution: save it; return
    for each option available from here:
        if option is valid:
            make the move          ← change the state
            solve(new state)
            undo the move          ← BACKTRACK
```

**Topics covered**
1. N-Queens: brute-force safety check
2. N-Queens: optimal with hashing (O(1) safety check)
3. Rat in a Maze

## 1. N-Queens

Place `n` queens on an `n × n` chessboard so that **no two queens attack each other** (no two share a row, a column, or a diagonal). Return every valid board.

**Idea:** place one queen per **column**, left to right. For column `col`, try every `row`; if that square is safe, place the queen and recurse to column `col + 1`. If no row works, return (backtrack) and move the previous queen.

Since we fill column by column, we only need to check the squares to the **left** of the current one:
- the same row, going left
- the upper-left diagonal
- the lower-left diagonal

In [1]:
class SolutionBrute:
    def is_safe(self, row, col, board, n):
        r, c = row, col
        while r >= 0 and c >= 0:                 # upper-left diagonal
            if board[r][c] == "Q":
                return False
            r -= 1
            c -= 1
        r, c = row, col
        while c >= 0:                            # same row, to the left
            if board[r][c] == "Q":
                return False
            c -= 1
        r, c = row, col
        while r < n and c >= 0:                  # lower-left diagonal
            if board[r][c] == "Q":
                return False
            r += 1
            c -= 1
        return True

    def solve(self, col, board, ans, n):
        if col == n:                             # all queens placed
            ans.append(list(board))              # copy the board
            return
        for row in range(n):
            if self.is_safe(row, col, board, n):
                board[row] = board[row][:col] + "Q" + board[row][col + 1:]   # place
                self.solve(col + 1, board, ans, n)
                board[row] = board[row][:col] + "." + board[row][col + 1:]   # remove

    def solve_n_queens(self, n):
        ans = []
        board = ["." * n for _ in range(n)]
        self.solve(0, board, ans, n)
        return ans

solutions = SolutionBrute().solve_n_queens(4)
print(len(solutions), "solutions for n = 4")
for board in solutions:
    print("\n".join(board), end="\n\n")

2 solutions for n = 4
..Q.
Q...
...Q
.Q..

.Q..
...Q
Q...
..Q.



**Time:** each safety check costs O(n), so the total is roughly O(n! · n). **Space:** O(n²) for the board.

## 2. N-Queens, optimised with hashing

Scanning the board for every check wastes time. Instead keep three arrays of flags and check **in O(1)**:

| Array | Size | What it tracks | Index for `(row, col)` |
|---|---|---|---|
| `left_row` | `n` | is this **row** already used? | `row` |
| `lower_diag` | `2n − 1` | "↙ / ↗" diagonals (anti-diagonals) | `row + col` |
| `upper_diag` | `2n − 1` | "↖ / ↘" diagonals (main diagonals) | `(n − 1) + (col − row)` |

**Why these indexes work**
- Every square on the same `↗` diagonal has the same `row + col`. In a 4×4 board it ranges from 0 to 6, so `2n − 1 = 7` slots are needed.
- Every square on the same `↘` diagonal has the same `col − row`. That ranges from `−(n−1)` to `(n−1)`; adding `n − 1` shifts it to `0 … 2n−2`.

Example: `row = 6, col = 3, n = 10` → `row + col = 9` and `(n−1) + (col−row) = 9 + (3−6) = 6`.

When placing a queen set all three flags to `1`; when backtracking set them back to `0`.

**Time:** O(n!) · **Space:** O(n) for the flag arrays (plus the board)

In [2]:
class SolutionOptimal:
    def solve(self, col, board, ans, left_row, upper_diag, lower_diag, n):
        if col == n:
            ans.append(board[:])
            return
        for row in range(n):
            if (left_row[row] == 0
                    and lower_diag[row + col] == 0
                    and upper_diag[n - 1 + col - row] == 0):
                board[row] = board[row][:col] + "Q" + board[row][col + 1:]
                left_row[row] = lower_diag[row + col] = upper_diag[n - 1 + col - row] = 1      # place

                self.solve(col + 1, board, ans, left_row, upper_diag, lower_diag, n)

                board[row] = board[row][:col] + "." + board[row][col + 1:]
                left_row[row] = lower_diag[row + col] = upper_diag[n - 1 + col - row] = 0      # backtrack

    def solve_n_queens(self, n):
        ans = []
        board = ["." * n for _ in range(n)]
        self.solve(0, board, ans, [0] * n, [0] * (2 * n - 1), [0] * (2 * n - 1), n)
        return ans

print(SolutionOptimal().solve_n_queens(4))

# Both versions must agree, and the solution counts for n = 1..8 are famous: 1, 0, 0, 2, 10, 4, 40, 92
counts = []
for n in range(1, 9):
    a, b = SolutionBrute().solve_n_queens(n), SolutionOptimal().solve_n_queens(n)
    assert a == b
    counts.append(len(b))
print(counts)
assert counts == [1, 0, 0, 2, 10, 4, 40, 92]

[['..Q.', 'Q...', '...Q', '.Q..'], ['.Q..', '...Q', 'Q...', '..Q.']]
[1, 0, 0, 2, 10, 4, 40, 92]


## 3. Rat in a Maze

A rat starts at the top-left `(0, 0)` of an `n × n` grid and must reach the bottom-right `(n−1, n−1)`.
- `1` = open cell, `0` = blocked cell
- It can move **Down (D), Left (L), Right (R), Up (U)**
- It may not visit the same cell twice **on the same path**

Return **all** paths in **lexicographically smallest order**. If we try the moves in alphabetical order **D → L → R → U**, the paths come out already sorted.

**Idea**
1. Mark the current cell as visited (`vis[i][j] = 1`).
2. Try each of the four directions: if the next cell is inside the grid, open, and not visited → recurse with the move letter appended.
3. After all four directions, **unmark** the cell (`vis[i][j] = 0`). This is the backtrack step: another path may use this cell later.

**Time:** O(4^(n²)) worst case · **Space:** O(n²) for `vis` plus recursion depth

> 🐞 **Bug fixed:** the first version unmarked a cell (`vis[i][j] = 0`) only *inside* a direction branch that was actually taken. A **dead-end cell** (no valid move) never ran any branch, so it stayed marked as visited forever and blocked every later path through it. Marking once before the loop and unmarking once **after** the loop always restores the cell.

In [3]:
class Maze:
    def find_paths(self, i, j, a, n, ans, path, vis):
        if i == n - 1 and j == n - 1:                # reached the destination
            ans.append(path)
            return
        vis[i][j] = 1                                # mark current cell
        for move, di, dj in (("D", 1, 0), ("L", 0, -1), ("R", 0, 1), ("U", -1, 0)):   # alphabetical order
            ni, nj = i + di, j + dj
            if 0 <= ni < n and 0 <= nj < n and not vis[ni][nj] and a[ni][nj] == 1:
                self.find_paths(ni, nj, a, n, ans, path + move, vis)
        vis[i][j] = 0                                # BACKTRACK: free the cell again

    def rat_maze(self, matrix):
        n = len(matrix)
        ans = []
        vis = [[0] * n for _ in range(n)]
        if matrix[0][0] == 1 and matrix[n - 1][n - 1] == 1:   # start and goal must both be open
            self.find_paths(0, 0, matrix, n, ans, "", vis)
        return ans

maze = [[1, 0, 0, 0],
        [1, 1, 0, 1],
        [1, 1, 0, 0],
        [0, 1, 1, 1]]
print(Maze().rat_maze(maze))                       # ['DDRDRR', 'DRDDRR']

['DDRDRR', 'DRDDRR']


> ✅ **Result:** the earlier version printed only `['DDRDRR']`, but there are **two** valid paths. `DRDDRR` goes (0,0)→(1,0)→(1,1)→(2,1)→(3,1)→(3,2)→(3,3). The old code missed it because cell (1,1) is a dead end when entered from (2,1), so it was left marked as visited (see the bug note above). Let's verify the answer by walking each path ourselves.

In [4]:
def check_path(maze, path):
    """Walk the path from (0,0) and verify every cell is open and never repeated."""
    n = len(maze)
    i = j = 0
    seen = {(0, 0)}
    for step in path:
        di, dj = {"D": (1, 0), "L": (0, -1), "R": (0, 1), "U": (-1, 0)}[step]
        i, j = i + di, j + dj
        assert 0 <= i < n and 0 <= j < n and maze[i][j] == 1, "left the maze or hit a wall"
        assert (i, j) not in seen, "revisited a cell"
        seen.add((i, j))
    return (i, j) == (n - 1, n - 1)

paths = Maze().rat_maze(maze)
print([check_path(maze, p) for p in paths])                 # [True, True]
assert paths == sorted(paths)                               # lexicographic order
print(Maze().rat_maze([[1, 1], [1, 1]]))                    # ['DR', 'RD']
print(Maze().rat_maze([[1, 0], [0, 1]]))                    # []  (no path)

[True, True]
['DR', 'RD']
[]


---
### ✅ Key takeaways
- Backtracking = **choose → explore → un-choose**.
- N-Queens: one queen per column; `row`, `row + col` and `(n−1) + col − row` identify the row and both diagonals in O(1).
- Rat in a Maze: mark the cell on the way in, **unmark on the way out**; try directions in alphabetical order for sorted output.
- Always verify with an independent check (counts for N-Queens, a path walker for the maze).